In [1]:
!pip install dask[complete]

In [2]:
import numpy as np
import pandas as pd
import dask.dataframe as dd

In [3]:
n = 5_000_000
df = pd.DataFrame({
    'id':np.arange(n),
    'value':np.random.normal(100,20,size=n),
    'category':np.random.choice(['A','B','C'],size=n)
})
df.to_csv('big_data.csv',index=False)

In [4]:
#czytanie big_data
ddf = dd.read_csv('big_data.csv')
print(ddf.head())

   id       value category
0   0   92.488979        B
1   1   90.119973        A
2   2   84.589716        A
3   3   87.896037        C
4   4  102.792694        C


In [5]:
#proste operacje
mean_val = ddf['value'].mean()
print(f"średnia leniwa: {mean_val}")
print(f"Średnia wartość: {mean_val.compute()}")

średnia leniwa: <dask_expr.expr.Scalar: expr=ArrowStringConversion(frame=FromMapProjectable(b49576c))['value'].mean(), dtype=float64>
Średnia wartość: 100.0134668375063


In [6]:
from dask.dataframe.backends import group_split_pandas
#wyznaczanie średnich dla kategorii: A,B,C
grouped = ddf.groupby('category')["value"].mean()
print(f"średnia leniwa: {grouped}")
print(f"Średnia wartość: {grouped.compute()}")

średnia leniwa: Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 5 expressions
Expr=((ArrowStringConversion(frame=FromMapProjectable(b49576c))[['value', 'category']]).mean(observed=False, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='value'))['value']
Średnia wartość: category
A    100.007175
B     99.985474
C    100.047744
Name: value, dtype: float64


In [8]:
print(f"liczba partycji: {ddf.npartitions}")
print(f"rozmiar danych szacowany: {ddf.memory_usage(deep=True).compute()}")

liczba partycji: 2
rozmiar danych szacowany: Index            264
id          40000000
value       40000000
category    45000000
dtype: int64
